# Word2Vec ConShift results

Loads outputs from `code/anchors_word2vec_conshift.py` in `results/anchors_word2vec/`:

- **`*_meta.txt`** — run settings and vocabulary sizes  
- **`*_shifts.tsv`** — each shared word, shift score, and **training token counts** in source/target corpus (`count_<corpus>` columns when you re-run the script)  
- **`*_anchors.txt`** — anchor words (lowest-shift quantile)  
- **`anchors_triple_intersection.txt`** — optional; anchors common to all three pairs  

**Top-N tables** are **one table per pair** with columns `word`, `shift`, `count_a`, `count_b` (source = a, target = b). Counts come from the TSV if present, else from `embeddings/word2vec/*.model`.

**Working directory:** the first code cell resolves the repo root and `results/anchors_word2vec`.

In [ ]:
from pathlib import Path
import pandas as pd

TOP_N = 10
CORPORA = ["arxiv", "yelp", "ciao"]

candidates_res = [
    Path("results/anchors_word2vec"),
    Path("../results/anchors_word2vec"),
    Path("../../results/anchors_word2vec"),
]
RESULTS_DIR = next((p.resolve() for p in candidates_res if p.is_dir()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError(
        "Could not find results/anchors_word2vec. "
        "cd to repo root or notebooks/ and re-run."
    )

candidates_repo = [RESULTS_DIR.parent, RESULTS_DIR.parent.parent]
REPO_ROOT = next(
    (p for p in candidates_repo if (p / "embeddings" / "word2vec").is_dir()),
    RESULTS_DIR.parent,
)
print(f"RESULTS_DIR = {RESULTS_DIR}")
print(f"REPO_ROOT   = {REPO_ROOT}")

In [ ]:
def load_meta(meta_path: Path) -> dict:
    out = {}
    for line in meta_path.read_text(encoding="utf-8").splitlines():
        if "\t" in line:
            k, v = line.split("\t", 1)
            out[k] = v
    out["pair"] = meta_path.name.replace("_meta.txt", "")
    return out


def parse_pair(pair_name: str):
    a, b = pair_name.split("_to_", 1)
    return a, b


_wv_cache: dict = {}


def wv_word_count(wv, w: str) -> int:
    if hasattr(wv, "get_vecattr"):
        try:
            c = wv.get_vecattr(w, "count")
            if c is not None:
                return int(c)
        except (KeyError, ValueError, TypeError):
            pass
    vocab = getattr(wv, "vocab", None)
    if vocab is not None and w in vocab:
        return int(vocab[w].count)
    return 0


def get_wv(corpus: str):
    if corpus not in _wv_cache:
        from gensim.models import Word2Vec

        path = REPO_ROOT / "embeddings" / "word2vec" / f"{corpus}.model"
        if not path.is_file():
            raise FileNotFoundError(f"Missing Word2Vec model: {path}")
        m = Word2Vec.load(str(path))
        _wv_cache[corpus] = m.wv
    return _wv_cache[corpus]


def attach_pair_counts(df: pd.DataFrame, src: str, tgt: str) -> pd.DataFrame:
    cs, ct = f"count_{src}", f"count_{tgt}"
    if cs in df.columns and ct in df.columns:
        return df
    wv_a, wv_b = get_wv(src), get_wv(tgt)
    out = df.copy()
    if cs not in out.columns:
        out[cs] = [wv_word_count(wv_a, w) for w in out["word"]]
    if ct not in out.columns:
        out[ct] = [wv_word_count(wv_b, w) for w in out["word"]]
    return out


def load_shifts(tsv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(tsv_path, sep="\t")
    pair_name = tsv_path.name.replace("_shifts.tsv", "")
    df["pair"] = pair_name
    src, tgt = parse_pair(pair_name)
    return attach_pair_counts(df, src, tgt)


def count_columns_for_pair(pair_name: str) -> list[str]:
    src, tgt = parse_pair(pair_name)
    return [f"count_{src}", f"count_{tgt}"]


def pairwise_top_n(
    df: pd.DataFrame, pair_name: str, top_n: int, *, largest: bool
) -> pd.DataFrame:
    """Columns: rank, word, shift, count_a, count_b (a=source, b=target)."""
    src, tgt = parse_pair(pair_name)
    ca, cb = f"count_{src}", f"count_{tgt}"
    part = (
        df.nlargest(top_n, "shift")
        if largest
        else df.nsmallest(top_n, "shift")
    )
    out = part[["word", "shift", ca, cb]].copy()
    out.insert(0, "rank", range(1, top_n + 1))
    return out.rename(columns={ca: "count_a", cb: "count_b"})


shift_files = sorted(RESULTS_DIR.glob("*_shifts.tsv"))
meta_files = sorted(RESULTS_DIR.glob("*_meta.txt"))
anchor_files = sorted(RESULTS_DIR.glob("*_anchors.txt"))
print("Shift files:", [p.name for p in shift_files])
print("Meta files:", [p.name for p in meta_files])

## Pairwise metadata

In [ ]:
meta_df = pd.DataFrame([load_meta(p) for p in meta_files])
if not meta_df.empty:
    display(meta_df.set_index("pair"))

## Top *N* most shifted (highest shift) per pair

One table **per pair** (`arxiv_to_ciao`, `arxiv_to_yelp`, `yelp_to_ciao`). Columns: **`word`**, **`shift`**, **`count_a`**, **`count_b`**, where **a** = source corpus and **b** = target corpus in the pair name. Set `TOP_N` in the first cell.

In [ ]:
from IPython.display import Markdown, display

for p in shift_files:
    df = load_shifts(p)
    pair_name = df["pair"].iloc[0]
    src, tgt = parse_pair(pair_name)
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=True))

## Top *N* most stable (lowest shift) per pair

Same as above: **three tables**, columns `word`, `shift`, `count_a`, `count_b`.

In [ ]:
from IPython.display import Markdown, display

for p in shift_files:
    df = load_shifts(p)
    pair_name = df["pair"].iloc[0]
    src, tgt = parse_pair(pair_name)
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=False))

## Words in all three pairwise vocabularies

Merge shift scores on `word` and summarize **mean shift** across the three pairs. Tables include **counts** `count_arxiv`, `count_yelp`, `count_ciao` (from Word2Vec training stats).

In [ ]:
if len(shift_files) >= 2:
    merged = None
    for p in shift_files:
        df = pd.read_csv(p, sep="\t")
        pair = p.name.replace("_shifts.tsv", "")
        sub = df.rename(columns={"shift": f"shift_{pair}"})[["word", f"shift_{pair}"]]
        merged = sub if merged is None else merged.merge(sub, on="word", how="inner")
    shift_cols = [c for c in merged.columns if c.startswith("shift_")]
    merged["mean_shift"] = merged[shift_cols].mean(axis=1)
    for c in CORPORA:
        col = f"count_{c}"
        if col not in merged.columns:
            wv = get_wv(c)
            merged[col] = [wv_word_count(wv, w) for w in merged["word"]]
    count_cols = [f"count_{c}" for c in CORPORA]
    print(f"Words in all {len(shift_files)} pairwise vocabularies: {len(merged)}")
    show_cols = ["word", "mean_shift"] + shift_cols + count_cols
    display(merged.nlargest(TOP_N, "mean_shift")[show_cols])
    display(merged.nsmallest(TOP_N, "mean_shift")[show_cols])
else:
    print("Need at least two shift files for merge.")

## Anchor list sizes

In [ ]:
anchor_counts = []
for p in anchor_files:
    n = sum(1 for _ in p.open(encoding="utf-8"))
    anchor_counts.append({"pair": p.name.replace("_anchors.txt", ""), "anchor_lines": n})
display(pd.DataFrame(anchor_counts))

## Triple intersection anchors (if generated)

Created with `python code/anchors_word2vec_conshift.py --mode triple-intersection`.

In [ ]:
triple_path = RESULTS_DIR / "anchors_triple_intersection.txt"
if triple_path.is_file():
    words = [ln.strip() for ln in triple_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    print(f"Triple intersection count: {len(words)}")
    display(pd.DataFrame({"word": words[:200]}))  # preview
else:
    print(f"No file at {triple_path}")

## Full shift tables (optional)

Uncomment to load every pairwise file into one long dataframe (can be large).

In [ ]:
# all_shifts = pd.concat([load_shifts(p) for p in shift_files], ignore_index=True)
# display(all_shifts.groupby("pair")["shift"].describe())